# Comparing Semgrep Community Rules vs Custom Rules for Python Security Scanning

**Purpose:** Compare the output, coverage, and workflow differences between using Semgrep's built-in community rule registry and writing custom rules for Python security scanning.

This notebook walks through scanning the same Python target with both approaches and compares the findings.

## When to use

- **Community rules** — quick start, broad coverage across common vulnerability patterns, no maintenance burden. Use when onboarding a new codebase or establishing a baseline.
- **Custom rules** — targeted detection for project-specific patterns, internal libraries, or organizational policies not covered by the registry. Use when community rules produce too many false negatives for your stack.
- **Both together** — run community rules for baseline coverage, then layer custom rules on top for gaps. This is the recommended production pattern.

## Prerequisites

- Semgrep CLI installed (`pip install semgrep`)
- A Python project or repository to scan
- Write access to create rule files in the working directory

In [ ]:
import subprocess
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve()
SCAN_TARGET = str(REPO_ROOT)
print(f"Scan target: {SCAN_TARGET}")

In [ ]:
# Verify semgrep is installed
result = subprocess.run(["semgrep", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print("semgrep not found. Install with: pip install semgrep")
    sys.exit(1)
print(f"Semgrep version: {result.stdout.strip()}")

## Step 1 — Scan with community rules

Community rules come from the Semgrep Registry and cover a wide range of security patterns across many languages and frameworks. The `--config=auto` preset selects a sensible default set of rules.

In [ ]:
print("Running semgrep scan with community rules (--config=auto)...\n")
community_result = subprocess.run(
    [
        "semgrep", "scan",
        "--config=auto",
        "--json",
        "--quiet",
        "--exclude", "node_modules",
        "--exclude", ".git",
        "--exclude", "__pycache__",
        "--exclude", "venv",
        "--max-target-bytes", "50000",
        SCAN_TARGET,
    ],
    capture_output=True, text=True, timeout=120,
)

if community_result.returncode not in (0, 1):
    print(f"Community scan failed (exit {community_result.returncode})")
    print(community_result.stderr[:500])
    sys.exit(1)

community_data = json.loads(community_result.stdout) if community_result.stdout else {}
community_findings = community_data.get("results", [])
print(f"Community rules findings: {len(community_findings)}")

# Show a summary of rule IDs found
rule_counts = {}
for finding in community_findings:
    rule_id = finding.get("check_id", "unknown")
    rule_counts[rule_id] = rule_counts.get(rule_id, 0) + 1

for rule_id, count in sorted(rule_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {rule_id}: {count}")

## Step 2 — Create custom rules

Custom rules let you target patterns that community rules miss. Below is a minimal custom rule that detects `pickle.loads` usage — a common Python security anti-pattern for deserialization.

In [ ]:
custom_rule = {
    "rules": [
        {
            "id": "custom-pickle-deserialization",
            "pattern": "pickle.loads(...)",
            "message": "Deserializing untrusted input with pickle is unsafe. Consider using json.loads or a signed serializer.",
            "languages": ["python"],
            "severity": "WARNING",
        },
        {
            "id": "custom-subprocess-shell-true",
            "pattern": "subprocess.run(..., shell=True)",
            "message": "Using shell=True in subprocess.run introduces shell injection risk. Pass arguments as a list instead.",
            "languages": ["python"],
            "severity": "ERROR",
        },
    ]
}

rule_dir = REPO_ROOT / "semgrep-custom-rules"
rule_dir.mkdir(exist_ok=True)
rule_file = rule_dir / "custom-python-rules.yaml"
with open(rule_file, "w") as f:
    json.dump(custom_rule, f, indent=2)
print(f"Custom rules written to {rule_file}")

## Step 3 — Scan with custom rules

Run Semgrep against the same target using only the custom rules defined above.

In [ ]:
print("Running semgrep scan with custom rules...\n")
custom_result = subprocess.run(
    [
        "semgrep", "scan",
        f"--config={rule_file}",
        "--json",
        "--quiet",
        "--exclude", "node_modules",
        "--exclude", ".git",
        "--exclude", "__pycache__",
        "--exclude", "venv",
        "--max-target-bytes", "50000",
        SCAN_TARGET,
    ],
    capture_output=True, text=True, timeout=120,
)

if custom_result.returncode not in (0, 1):
    print(f"Custom scan failed (exit {custom_result.returncode})")
    print(custom_result.stderr[:500])
    sys.exit(1)

custom_data = json.loads(custom_result.stdout) if custom_result.stdout else {}
custom_findings = custom_data.get("results", [])
print(f"Custom rules findings: {len(custom_findings)}")

for finding in custom_findings:
    print(f"  [{finding.get('check_id')}] {finding.get('path')}:{finding.get('start', {}).get('line')}")

## Step 4 — Compare results

The comparison below highlights what each approach catches. Community rules provide broad coverage; custom rules fill project-specific gaps.

In [ ]:
community_ids = set(f.get("check_id", "") for f in community_findings)
custom_ids = set(f.get("check_id", "") for f in custom_findings)

print(f"Community rule IDs found: {len(community_ids)}")
print(f"Custom rule IDs found: {len(custom_ids)}")
print(f"Overlap (IDs in both): {len(community_ids & custom_ids)}")
print(f"Community-only: {len(community_ids - custom_ids)}")
print(f"Custom-only: {len(custom_ids - community_ids)}")

if custom_ids - community_ids:
    print("\nCustom-only detections (gaps community rules miss):")
    for rid in sorted(custom_ids - community_ids):
        print(f"  - {rid}")

print("\nRecommendation: Use community rules for baseline coverage, then add custom rules for project-specific patterns.")

## Verify

Confirm that both scans completed without fatal errors and that the custom rule file was created correctly.

In [ ]:
assert rule_file.exists(), "Custom rule file was not created"
assert community_result.returncode in (0, 1), "Community scan exited with unexpected code"
assert custom_result.returncode in (0, 1), "Custom scan exited with unexpected code"
print("All checks passed.")

## Common errors

| Error | Cause | Fix |
|---|---|---|
| `semgrep: command not found` | Semgrep not installed or not on PATH | `pip install semgrep` and verify with `semgrep --version` |
| Exit code 2 with no findings | No Python files in scan target | Ensure the target directory contains `.py` files |
| Custom rule returns 0 findings | Rule pattern does not match code | Test the rule pattern against a known vulnerable file first |
`--max-target-bytes` exceeded | Large files skipped | Increase the limit or exclude generated files

## References

Semgrep Registry and CLI documentation are available at semgrep.dev. Custom rule syntax is documented in the Semgrep rule-writing guide.